# Week 14 - Vision-Language Models and Modern AI Vision

**MCTE 4323 / MCTA 4364 Machine Vision**

### Learning objectives
By the end of this lab you will be able to:
- Explain how **vision-language models (VLMs)** connect images and text in a shared space.
- Perform **zero-shot classification** with **CLIP** using your own text labels.
- Generate **image captions** and answer **visual questions** with a VLM.
- Discuss **prompting**, **hallucination**, **bias**, and deployment limitations.

### Why this matters
Instead of training a classifier for a fixed label set, VLMs let you describe the task in **natural language**. This is transforming industrial inspection, assistive tech and robotics.

## 1. Setup
> This notebook downloads large pretrained models (hundreds of MB). A **GPU runtime** is strongly recommended, and it is **optional** - the markdown and concept maps run without it.

In [ ]:
import os
if not os.path.isdir("MCTA-4364-Machine-Vision"):
    !git clone https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git
%cd MCTA-4364-Machine-Vision
!pip -q install torch transformers pillow matplotlib opencv-python ipywidgets

In [ ]:
import sys
sys.path.append("resources/scripts")
import cv2, numpy as np, torch
from PIL import Image
from cvhelpers import show, concept_map
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"

## 2. How a VLM works

In [ ]:
concept_map([
    "Image -> vision encoder (CNN or ViT)",
    "Text -> language encoder (transformer)",
    "Project both into a shared embedding space (contrastive loss)",
    "Tasks: zero-shot classification, captioning, VQA, grounding",
    "Prompt with natural language instead of retraining"
], title="Vision-language model pipeline")

## 3. Guided example - zero-shot classification with CLIP
CLIP scores how well an image matches each of several **text prompts**. No training is needed for new classes - you simply write new prompts.

In [ ]:
from transformers import CLIPProcessor, CLIPModel

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("CLIP loaded.")

In [ ]:
image = Image.open("resources/images/test_image.jpeg").convert("RGB")
labels = ["a photo of a person", "a photo of a car", "a photo of an animal",
          "a photo of food", "a photo of machinery"]

inputs = clip_processor(text=labels, images=image, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    probs = clip_model(**inputs).logits_per_image.softmax(dim=1)[0].cpu().tolist()

for label, p in sorted(zip(labels, probs), key=lambda x: -x[1]):
    print(f"{p * 100:5.1f}%  {label}")
show(cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR), titles=["Query image"])

###  Interactive exploration - write your own prompts
Edit the labels (comma-separated) and press Enter. Notice how the wording changes the result - this is **prompt engineering**.

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def clip_demo(labels_text="a photo of a person, a photo of a car, a photo of a tree, a photo of a building"):
    lab = [l.strip() for l in labels_text.split(",") if l.strip()]
    inp = clip_processor(text=lab, images=image, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        pr = clip_model(**inp).logits_per_image.softmax(dim=1)[0].cpu().tolist()
    for l, p in sorted(zip(lab, pr), key=lambda x: -x[1]):
        print(f"{p * 100:5.1f}%  {l}")

interact(clip_demo, labels_text=widgets.Text(value="a photo of a person, a photo of a car, a photo of a tree",
                                             description="labels", layout=widgets.Layout(width="90%")))

## 4. Guided example - image captioning with BLIP
An image-to-text model describes the scene in natural language.

In [ ]:
try:
    from transformers import BlipProcessor, BlipForConditionalGeneration
    blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    blip_model = BlipForConditionalGeneration.from_pretrained(
        "Salesforce/blip-image-captioning-base").to(device).eval()
    inputs = blip_processor(image, return_tensors="pt").to(device)
    with torch.no_grad():
        out = blip_model.generate(**inputs, max_new_tokens=30)
    print("Caption:", blip_processor.decode(out[0], skip_special_tokens=True))
except Exception as e:
    print("Captioning demo skipped:", e)

## 5. Guided example - visual question answering (VQA)
Ask a natural-language question about the image. **Prompt design** controls the answer style.

In [ ]:
try:
    from transformers import BlipProcessor, BlipForQuestionAnswering
    vqa_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
    vqa_model = BlipForQuestionAnswering.from_pretrained(
        "Salesforce/blip-vqa-base").to(device).eval()
    for question in ["What is in the image?", "How many people are there?"]:
        inp = vqa_processor(image, question, return_tensors="pt").to(device)
        with torch.no_grad():
            out = vqa_model.generate(**inp, max_new_tokens=10)
        print(f"Q: {question}\nA: {vqa_processor.decode(out[0], skip_special_tokens=True)}\n")
except Exception as e:
    print("VQA demo skipped:", e)

## 6. The modern VLM landscape

| Model family | Strength | Typical use |
|---|---|---|
| CLIP / SigLIP | image-text matching | zero-shot classification, retrieval |
| BLIP / BLIP-2 | captioning, VQA | accessibility, description |
| LLaVA / Qwen2-VL | chat-style reasoning | inspection assistant, robotics |
| Grounding DINO + SAM | text-driven detection + masks | open-vocabulary inspection |

**Prompt-driven vision** lets a single model serve many tasks, but always validate the output.

## 7. Exercise (complete the code)

1. Add a **negative prompt set** to the CLIP demo (e.g. *"a photo of a cat"*) and check whether the correct label still wins.
2. Test the captioning model on `resources/images/money_counter.png` and on `resources/images/messi.jpg`.
3. Record one confident but **wrong** prediction. This is the kind of failure you must handle in deployment.

In [ ]:
# TODO: CLIP with negative prompts and captioning tests


## 8. Challenge (independent)

Design a **factory inspection assistant** using a VLM: capture an image of a PCB, ask *"Is there a missing component?"*, and log the answer. Then list the risks (hallucination, latency, privacy, model bias) and propose one safeguard for each.

In [ ]:
# Your code here


## 9. Check your understanding (Q&A)

<details><summary><b>Q1. What does 'zero-shot' mean in CLIP?</b></summary>

The model classifies images into classes it was never explicitly trained on, by matching the image against text prompts in a shared embedding space.
</details>

<details><summary><b>Q2. What is a hallucination in a VLM?</b></summary>

A confident, fluent output that is not supported by the image (e.g. inventing an object that is not there). It happens because the language model favours plausible text.
</details>

<details><summary><b>Q3. Why might a VLM be unsuitable for a hard real-time safety system?</b></summary>

Large models are slow, can be non-deterministic, may hallucinate, and are hard to certify. Real-time safety systems usually need smaller, deterministic, validated models.
</details>

<details><summary><b>Q4. Name one way to reduce hallucination without retraining.</b></summary>

Ask the model to answer only from the image, request a confidence, constrain the answer format, or combine it with a detector/classifier whose outputs are grounded.
</details>

## 10. Further reading & self-exploration
- CLIP paper and model card: https://huggingface.co/openai/clip-vit-base-patch32
- BLIP model card: https://huggingface.co/Salesforce/blip-image-captioning-base
- Hugging Face vision-language models: https://huggingface.co/docs/transformers/tasks/image_captioning
- LLaVA: https://llava-vl.github.io/
- Grounding DINO: https://github.com/IDEA-Research/GroundingDINO
- Wikipedia - Vision-language model: https://en.wikipedia.org/wiki/Vision-language_model

**Try next:** build a small retrieval system that finds the most similar image to a text query using CLIP embeddings.

## 11. Key takeaways
- VLMs align images and text in a shared embedding space.
- CLIP enables **zero-shot** classification via **prompts**.
- BLIP/LLaVA add captioning and visual question answering.
- Prompt wording strongly affects results.
- Always handle **hallucination, bias, latency and privacy** in deployment.